In [ ]:
from neuroCombat import neuroCombat
from onekey_algo import get_param_in_cwd
import pandas as pd
import numpy as np

feature_file = r'G:\WHO grading\Zhongyi\模型组件\传统组学\features\rad_features批次矫正前.csv'   # 影像组学/深度学习特征
label_file   = r'G:\WHO grading\Zhongyi\模型组件\传统组学\features\label-RND-0.csv'
clinic_file  = r'"G:\WHO grading\Zhongyi\模型组件\传统组学\features\clinic.csv"'
rad_data = pd.read_csv(feature_file)


ids_ = rad_data['ID']
rad_data = rad_data.drop('ID', axis=1)
label_data = pd.read_csv(label_file)
clinical_data = pd.read_csv(clinic_file)
covars = pd.merge(label_data, clinical_data, on='ID', how='inner')

data_combat = neuroCombat(dat=rad_data.T,
    covars=covars,
    batch_col=get_param_in_cwd('dataset_column', 'group'),
    categorical_cols=get_param_in_cwd('categorical_cols', []))
rad_data = pd.DataFrame(data_combat['data'].T, columns=rad_data.columns)
rad_data.insert(0, 'ID', ids_)
rad_data.to_csv(feature_file.replace('.csv', '_combat.csv'), index=False)
rad_data

In [ ]:
# -*- coding: utf-8 -*-
"""
ComBat 批次校正 + 形状+颜色双重区分 + 等比例圆形区域 PCA 图
- 左右图独立 PCA，解释率分别计算
- 每个批次分配唯一的颜色 + 唯一的形状（色盲友好）
- 坐标轴范围等比例（aspect='equal'），点云不被拉伸
- 自动居中，确保点集群位于图形中央
- 左右图均显示图例
"""

from neuroCombat import neuroCombat
from onekey_algo import get_param_in_cwd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings

# ==================== 用户配置区域 ====================
feature_file = r'G:\WHO grading\Zhongyi\模型组件\传统组学\features\rad_features批次矫正前.csv'   # 影像组学/深度学习特征
label_file   = r'G:\WHO grading\Zhongyi\模型组件\传统组学\features\label-RND-0.csv'
clinic_file  = r'G:\WHO grading\Zhongyi\模型组件\传统组学\features\clinic.csv'

BATCH_COL = get_param_in_cwd('dataset_column', 'group')
CATEGORICAL_COLS = get_param_in_cwd('categorical_cols', ['age', 'gender'])
PCA_SAVE_PATH = 'pca_comparison_circular.png'
# =====================================================

def get_color_mapping(unique_batches):
    """根据批次数量自动选择合适的颜色映射"""
    n = len(unique_batches)
    if n <= 10:
        cmap = plt.cm.tab10
        colors = [cmap(i) for i in range(n)]
    elif n <= 20:
        cmap = plt.cm.tab20
        colors = [cmap(i) for i in range(n)]
    else:
        base_cmap = plt.cm.tab20
        colors = [base_cmap(i % 20) for i in range(n)]
        warnings.warn(f"批次数量 ({n}) 超过20，颜色可能重复，建议合并或检查分组。")
    return {batch: colors[i] for i, batch in enumerate(unique_batches)}

def get_marker_mapping(unique_batches):
    """为每个批次分配唯一的标记形状"""
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', 'd', '|', '_']
    n = len(unique_batches)
    if n > len(markers):
        warnings.warn(f"批次数量 ({n}) 超过可用形状数量 ({len(markers)})，形状将循环使用，可能重复。")
    marker_mapping = {batch: markers[i % len(markers)] for i, batch in enumerate(unique_batches)}
    return marker_mapping

def plot_pca_corrected(df_before, df_after, batch_series, save_path):
    """
    增强版 PCA 对比图（颜色+形状双重区分 + 等比例圆形区域）：
    - 分别对校正前后数据独立做 PCA
    - 计算正方形视图范围（x和y范围长度相等），使点云不被拉伸
    - 启用 ax.set_aspect('equal') 确保显示比例真实
    - 自动居中，图例动态调整
    """
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.size'] = 12

    # 过滤缺失批次值的样本
    valid_mask = batch_series.notna()
    if not valid_mask.all():
        warnings.warn(f"发现 {(~valid_mask).sum()} 个样本批次缺失，已自动剔除。")
        batch_series = batch_series[valid_mask]
        df_before = df_before.loc[valid_mask]
        df_after = df_after.loc[valid_mask]

    unique_batches = batch_series.unique()
    color_mapping = get_color_mapping(unique_batches)
    marker_mapping = get_marker_mapping(unique_batches)

    print("批次 -> 颜色 & 形状 映射：")
    for batch in unique_batches:
        print(f"  {batch} : 颜色={color_mapping[batch]}, 形状={marker_mapping[batch]}")

    # ---- 校正前独立 PCA ----
    df_before_scaled = (df_before - df_before.mean()) / df_before.std()
    pca_before = PCA(n_components=2)
    comp_before = pca_before.fit_transform(df_before_scaled)
    var_before = pca_before.explained_variance_ratio_

    # ---- 校正后独立 PCA ----
    df_after_scaled = (df_after - df_after.mean()) / df_after.std()
    pca_after = PCA(n_components=2)
    comp_after = pca_after.fit_transform(df_after_scaled)
    var_after = pca_after.explained_variance_ratio_

    # ===== 计算正方形视图范围（等比例，使点云不被拉伸） =====
    # 合并所有点的坐标
    all_x = np.concatenate([comp_before[:, 0], comp_after[:, 0]])
    all_y = np.concatenate([comp_before[:, 1], comp_after[:, 1]])
    # 中心点（使用中位数）
    center_x = np.median(all_x)
    center_y = np.median(all_y)
    # 计算每个点到中心的切比雪夫距离（max norm），因为正方形区域
    half_width = np.max(np.abs(all_x - center_x))
    half_height = np.max(np.abs(all_y - center_y))
    # 取较大的半边长作为正方形半径，并增加10%余量
    radius = max(half_width, half_height) * 1.1
    if radius < 1e-6:
        radius = 1.0
    # 正方形边界
    x_min = center_x - radius
    x_max = center_x + radius
    y_min = center_y - radius
    y_max = center_y + radius
    # ===== 结束计算 =====

    # ---- 创建并排子图，调整宽高比以配合等比例坐标轴 ----
    # 由于要设置 aspect='equal'，子图的实际显示宽度会由数据范围决定。
    # 为了让并排图整体美观，可以手动调整 figsize 的高度和宽度比例。
    # 根据正方形边长和图形宽度比例，这里保持 figsize 宽高比为 2:1 左右，（两个正方形并排）
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # 绘制校正前散点图
    for batch in unique_batches:
        mask = batch_series == batch
        ax1.scatter(comp_before[mask, 0], comp_before[mask, 1],
                    color=color_mapping[batch], marker=marker_mapping[batch],
                    label=str(batch), alpha=0.8, edgecolor='white', linewidth=0.5, s=60)
    ax1.set_title('Before ComBat (Independent PCA)', fontsize=14, fontweight='bold')
    ax1.set_xlabel(f'PC1 ({var_before[0]:.1%})', fontsize=12)
    ax1.set_ylabel(f'PC2 ({var_before[1]:.1%})', fontsize=12)
    ax1.tick_params(labelsize=10)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    ax1.set_aspect('equal')   # 等比例，保证圆形区域

    # 绘制校正后散点图
    for batch in unique_batches:
        mask = batch_series == batch
        ax2.scatter(comp_after[mask, 0], comp_after[mask, 1],
                    color=color_mapping[batch], marker=marker_mapping[batch],
                    label=str(batch), alpha=0.8, edgecolor='white', linewidth=0.5, s=60)
    ax2.set_title('After ComBat (Independent PCA)', fontsize=14, fontweight='bold')
    ax2.set_xlabel(f'PC1 ({var_after[0]:.1%})', fontsize=12)
    ax2.set_ylabel(f'PC2 ({var_after[1]:.1%})', fontsize=12)
    ax2.tick_params(labelsize=10)
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(y_min, y_max)
    ax2.set_aspect('equal')

    # 图例动态放置（若批次过多则移到图外）
    if len(unique_batches) > 8:
        ax1.legend(title='Center', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
        ax2.legend(title='Center', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
    else:
        ax1.legend(title='Center', loc='best', frameon=True)
        ax2.legend(title='Center', loc='best', frameon=True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✅ PCA 对比图已保存至: {save_path} (等比例坐标轴，点云区域大致呈圆形，独立 PCA)")

# ========== 主程序（保持不变） ==========
if __name__ == "__main__":
    print("读取特征文件...")
    rad_data = pd.read_csv(feature_file)
    ids_ = rad_data['ID']
    rad_data = rad_data.drop('ID', axis=1)

    print("读取标签和临床文件...")
    label_data = pd.read_csv(label_file)
    clinical_data = pd.read_csv(clinic_file)
    covars = pd.merge(label_data, clinical_data, on='ID', how='inner')
    print(f"协变量表 shape: {covars.shape}")

    if BATCH_COL not in covars.columns:
        raise ValueError(f"协变量表中缺少批次列 '{BATCH_COL}'，请检查配置。")
    for col in CATEGORICAL_COLS:
        if col not in covars.columns:
            raise ValueError(f"协变量表中缺少保护列 '{col}'，请检查配置。")

    rad_data_before = rad_data.copy()
    batch_series = covars[BATCH_COL]

    print("正在执行 ComBat 校正...")
    data_combat = neuroCombat(
        dat=rad_data.T,
        covars=covars,
        batch_col=BATCH_COL,
        categorical_cols=CATEGORICAL_COLS
    )
    rad_data_combat = pd.DataFrame(data_combat['data'].T, columns=rad_data.columns)
    print("ComBat 校正完成。")

    print("绘制 PCA 对比图（等比例圆形区域，颜色+形状双重区分）...")
    plot_pca_corrected(rad_data_before, rad_data_combat, batch_series, PCA_SAVE_PATH)

    rad_data_output = rad_data_combat.copy()
    rad_data_output.insert(0, 'ID', ids_)
    output_file = feature_file.replace('.csv', '_combat.csv')
    rad_data_output.to_csv(output_file, index=False)
    print(f"✅ 校正后的特征已保存至: {output_file}")

    rad_data_output